# 03 — Full-year one-arm profile for the department desktop

This is the authoritative dense CPU benchmark. It uses the full-year EMRI
configuration and the physical `esa-trailing-orbits.h5` orbit file.

The waveform cache, WDM coefficient cache and profiling summaries are saved
directly beside the notebook with a `_local_wdm_` prefix. Notebook 04 reuses
the coefficient cache created here.

In [1]:
import gc
import os
import platform

# This notebook benchmarks the CPU implementation.
# These must be set before importing JAX, FEW, or packages that import them.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import jax.numpy as jnp

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform

import json
import platform
import subprocess
import time
from pathlib import Path

import jax
import numpy as np

from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

import WDM_LISAresponse


NOTEBOOK_DIR = Path.cwd().resolve()
ORBIT_FILE = NOTEBOOK_DIR / "esa-trailing-orbits.h5"

if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        "Place esa-trailing-orbits.h5 in the same directory as these notebooks. "
        f"Current notebook directory: {NOTEBOOK_DIR}"
    )


def load_lisa_orbits(path: Path):
    """Load the physical LISA orbit file used by the response calculations."""
    orbit_data = read_orbit_file(str(path))
    orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
    orbits = InterpolatedOrbits(
        orbit_times,
        orbit_data.positions,
        orbit_data.velocities,
    )
    return orbit_data, orbits


def git_state(package_file: str) -> dict[str, object]:
    """Return the current Git commit and whether the checkout is dirty."""
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(Path(package_file).resolve().parent),
            "rev-parse",
            "--show-toplevel",
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return {"root": None, "commit": None, "dirty": None}

    root = Path(completed.stdout.strip()).resolve()
    commit = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    dirty = bool(
        subprocess.run(
            ["git", "-C", str(root), "status", "--porcelain"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    )
    return {"root": str(root), "commit": commit, "dirty": dirty}


def json_ready(value):
    """Convert NumPy and JAX objects into JSON-compatible values."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item") and getattr(value, "ndim", None) == 0:
        return value.item()
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def save_json(filename: str, payload) -> Path:
    """Save a JSON result beside the notebook."""
    path = NOTEBOOK_DIR / filename
    path.write_text(
        json.dumps(json_ready(payload), indent=2),
        encoding="utf-8",
    )
    print("Saved:", path)
    return path


orbit_data, orbits = load_lisa_orbits(ORBIT_FILE)
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

print("Notebook directory:", NOTEBOOK_DIR)
print("Orbit file:", ORBIT_FILE)
print("Python:", platform.python_version())
print("JAX:", jax.__version__, "| backend:", jax.default_backend())
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("Source state:", git_state(WDM_LISAresponse.__file__))


from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMPlan
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_context_from_wdm,
    _build_wdm_dT_grid,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)

def block_result(value):
    """Synchronise every JAX leaf before stopping a timer."""
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def timed(function):
    started = time.perf_counter()
    value = function()
    block_result(value)
    return value, float(time.perf_counter() - started)


def timing_summary(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median_s": float(np.median(values)),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "all_s": [float(value) for value in values],
    }


def benchmark(function, *, warmups: int, repeats: int):
    for _ in range(int(warmups)):
        block_result(function())

    samples = []
    latest = None
    for _ in range(int(repeats)):
        latest, elapsed = timed(function)
        samples.append(elapsed)
    return latest, timing_summary(samples)


def compare_arrays(candidate, reference, *, m_start: int = 0):
    """Return max-absolute and relative-L2 differences."""
    candidate = jax.numpy.asarray(candidate)[..., m_start:]
    reference = jax.numpy.asarray(reference)[..., m_start:]
    difference = candidate - reference

    difference_power = jax.numpy.sum(jax.numpy.abs(difference) ** 2)
    reference_power = jax.numpy.sum(jax.numpy.abs(reference) ** 2)
    max_abs = jax.numpy.max(jax.numpy.abs(difference))
    rel_l2 = jax.numpy.where(
        reference_power > 0,
        jax.numpy.sqrt(difference_power / reference_power),
        jax.numpy.sqrt(difference_power),
    )
    block_result((max_abs, rel_l2))
    return {"max_abs": float(max_abs), "rel_l2": float(rel_l2)}


def compare_link_maps(candidate, reference, *, m_start: int = 0):
    if set(candidate) != set(reference):
        raise ValueError("Directed-link dictionaries contain different keys.")

    metrics = {
        str(link): compare_arrays(
            candidate[link],
            reference[link],
            m_start=m_start,
        )
        for link in reference
    }
    return {
        "per_link": metrics,
        "max_abs": max(item["max_abs"] for item in metrics.values()),
        "rel_l2": max(item["rel_l2"] for item in metrics.values()),
    }


process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    return (
        process.memory_info().rss / 1024**3
        if process is not None
        else float("nan")
    )


print("Machine:", platform.node())
print("CPU:", platform.processor())
print(
    "Physical cores:",
    psutil.cpu_count(logical=False) if psutil is not None else "unknown",
)


Notebook directory: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples
Orbit file: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/esa-trailing-orbits.h5
Python: 3.12.12
JAX: 0.9.2 | backend: cpu
WDM_LISAresponse: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
Source state: {'root': '/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse', 'commit': '6a5135635e74ece1beaf03beb140735534343640', 'dirty': True}
Machine: calx038.ast.cam.ac.uk
CPU: x86_64
Physical cores: 20


In [2]:
# Full-year production configuration

N = 3_200_000
dt = 10.0
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25
CALC_M0 = False

ROW_CHUNK_SIZE = 32_640
LAG_BLOCK_SIZE = 17

SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": ROW_CHUNK_SIZE,
    "lag_block_size": LAG_BLOCK_SIZE,
    "batch_chunk": 6,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

theta = np.pi / 2
phi = 0.0

DWT_WARMUPS = 2
DWT_REPEATS = 11
SHIFT_WARMUPS = 2
SHIFT_REPEATS = 11
PIPELINE_WARMUPS = 2
PIPELINE_REPEATS = 9

PARALLEL_SCHEDULES = [
    {"name": "groups_3_workers_2", "group_size": 3, "workers": 2},
    {"name": "groups_2_workers_3", "group_size": 2, "workers": 3},
    {"name": "groups_1_workers_6", "group_size": 1, "workers": 6},
]

SAVE_RESULTS = True
REUSE_CACHE = True


## 1. Local cache and physical inputs

The raw one-year waveform and WDM coefficients are large and may depend on
FEW, WDM and JAX versions. They are cached beside the notebook with
`_local_wdm_` filenames.

Delete the relevant cache files, or set `REUSE_CACHE=False`, after changing
the waveform generator, WDM transform, `calc_m0`, sampling or tiling.

In [3]:

source_state = git_state(WDM_LISAresponse.__file__)

waveform_cache = NOTEBOOK_DIR / "_local_wdm_03_waveform_cache.npz"
wdm_cache = NOTEBOOK_DIR / "_local_wdm_03_wdm_coefficients.npz"
cache_metadata = NOTEBOOK_DIR / "_local_wdm_03_cache_metadata.json"

# `orbit_data`, `orbits` and `t0` were loaded in the setup cell.
td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs={"pad_output": True, "odd_len": True},
    return_list=True,
)

model_kwargs = {
    "td_gen": td_gen,
    "m1": 1.0e6,
    "m2": 10.0,
    "a": 0.9,
    "p0": 8.5,
    "e0": 0.3,
    "x0": 1.0,
    "dist": 1.0,
    "theta": theta,
    "phi": phi,
    "qK": np.pi / 3,
    "phiK": np.pi / 3,
    "Phi_phi0": np.pi / 3,
    "Phi_theta0": 0.0,
    "Phi_r0": np.pi / 3,
    "eps": 1.0e-2,
}

if REUSE_CACHE and waveform_cache.exists():
    with np.load(waveform_cache, allow_pickle=False) as cached:
        times_pad = np.asarray(cached["times_pad"])
        h_plus = np.asarray(cached["h_plus"])
        h_cross = np.asarray(cached["h_cross"])
    waveform_generation_s = 0.0
    print("Reused waveform cache:", waveform_cache)
else:
    started = time.perf_counter()
    times_pad, h_plus, h_cross, padding = (
        generate_windowed_padded_waveform(
            waveform_model="emri",
            t0=t0,
            dt=dt,
            N=N,
            model_kwargs=model_kwargs,
            window_function="tukey",
            window_kwargs={"alpha": 0.005, "tukey_factor": 4.0},
            wdm_Nf=Nf,
            return_padding_info=True,
        )
    )
    waveform_generation_s = time.perf_counter() - started
    np.savez(
        waveform_cache,
        times_pad=times_pad,
        h_plus=h_plus,
        h_cross=h_cross,
    )
    print("Created waveform cache:", waveform_cache)

print("Padded samples:", times_pad.size)
print("Waveform generation_s:", f"{waveform_generation_s:.3f}")
print("RSS [GiB]:", f"{rss_gib():.2f}")


Created waveform cache: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_waveform_cache.npz
Padded samples: 3264000
Waveform generation_s: 8.770
RSS [GiB]: 6.53


In [4]:
# Build the WDM grid and reusable one-arm plan

grid_context, grid_s = timed(
    lambda: _build_wdm_dT_grid(
        times_pad,
        Nf=Nf,
        d=d_wdm,
        A_frac=A_frac,
        q=q_wdm,
        calc_m0=CALC_M0,
    )
)
one_arm_plan, plan_s = timed(
    lambda: OneArmWDMPlan.build_from_context(
        grid_context,
        orbits,
        theta,
        phi,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=SHIFT_KWARGS,
    )
)

print("WDM shape:", (grid_context["wdm"].Nt, grid_context["wdm"].Nf))
print("Grid build_s:", f"{grid_s:.3f}")
print("Plan build_s:", f"{plan_s:.3f}")
print(
    "Persistent plan memory [MiB]:",
    one_arm_plan.shift_plan.device_plan_memory_bytes / 1024**2,
)
print("RSS [GiB]:", f"{rss_gib():.2f}")


WDM shape: (32640, 100)
Grid build_s: 2.844
Plan build_s: 1.110
Persistent plan memory [MiB]: 153.14960861206055
RSS [GiB]: 7.09


## 2. WDM transforms and reusable local coefficient cache

In [5]:
def sequential_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=False,
    )


def parallel_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=True,
    )


sequential_context, sequential_dwt_stats = benchmark(
    sequential_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)
parallel_context, parallel_dwt_stats = benchmark(
    parallel_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)

dwt_plus = compare_arrays(
    parallel_context["w_t_plus"],
    sequential_context["w_t_plus"],
    m_start=1,
)
dwt_cross = compare_arrays(
    parallel_context["w_t_cross"],
    sequential_context["w_t_cross"],
    m_start=1,
)

# Preserve the real coefficient dtype used by the cleaned production path.
w_plus_host = np.asarray(parallel_context["w_t_plus"])
w_cross_host = np.asarray(parallel_context["w_t_cross"])

np.savez(
    wdm_cache,
    times_pad=times_pad,
    w_plus=w_plus_host,
    w_cross=w_cross_host,
    Nf=np.asarray(Nf),
    q_wdm=np.asarray(q_wdm),
    d_wdm=np.asarray(d_wdm),
    A_frac=np.asarray(A_frac),
    calc_m0=np.asarray(CALC_M0),
)
cache_metadata.write_text(
    json.dumps(
        json_ready(
            {
                "source": source_state,
                "configuration": {
                    "N": N,
                    "dt": dt,
                    "Nf": Nf,
                    "q_wdm": q_wdm,
                    "d_wdm": d_wdm,
                    "A_frac": A_frac,
                    "calc_m0": CALC_M0,
                },
                "shape": list(w_plus_host.shape),
                "dtype": str(w_plus_host.dtype),
            }
        ),
        indent=2,
    ),
    encoding="utf-8",
)

print("Sequential DWT:", sequential_dwt_stats)
print("Parallel DWT:", parallel_dwt_stats)
print("DWT comparisons:", dwt_plus, dwt_cross)
print("Saved WDM coefficient cache:", wdm_cache)


Sequential DWT: {'median_s': 0.21127481397707015, 'q25_s': 0.21055163099663332, 'q75_s': 0.2132206474780105, 'iqr_s': 0.0026690164813771844, 'minimum_s': 0.2087282069842331, 'maximum_s': 0.21466765902005136, 'all_s': [0.21116129198344424, 0.21466765902005136, 0.21427791198948398, 0.21008740400429815, 0.21127481397707015, 0.2122967870091088, 0.20950146799441427, 0.2110158579889685, 0.2115376210422255, 0.2141445079469122, 0.2087282069842331]}
Parallel DWT: {'median_s': 0.12061841698596254, 'q25_s': 0.11888866699882783, 'q75_s': 0.12367021600948647, 'iqr_s': 0.004781549010658637, 'minimum_s': 0.11713179398793727, 'maximum_s': 0.127227064978797, 'all_s': [0.12061841698596254, 0.12155245803296566, 0.1242981810355559, 0.12304225098341703, 0.127227064978797, 0.11849905800772831, 0.11713179398793727, 0.11997133999830112, 0.11927827598992735, 0.11786628799745813, 0.12660449498798698]}
DWT comparisons: {'max_abs': 0.0, 'rel_l2': 0.0} {'max_abs': 0.0, 'rel_l2': 0.0}
Saved WDM coefficient cache: /

## 3. Six-link scheduling sweep


In [6]:
w_plus = jnp.asarray(w_plus_host)
w_cross = jnp.asarray(w_cross_host)
block_result((w_plus, w_cross))

def materialized_shift():
    return one_arm_plan.apply_device(
        w_plus,
        w_cross,
        execution_mode="materialized",
    )


baseline_links, baseline_shift_stats = benchmark(
    materialized_shift,
    warmups=SHIFT_WARMUPS,
    repeats=SHIFT_REPEATS,
)
schedule_rows = [
    {
        "name": "materialized",
        "group_size": 6,
        "workers": 1,
        **baseline_shift_stats,
        "max_abs": 0.0,
        "rel_l2": 0.0,
    }
]

for schedule in PARALLEL_SCHEDULES:
    def run_parallel(schedule=schedule):
        return one_arm_plan.apply_device(
            w_plus,
            w_cross,
            execution_mode="materialized_parallel_links",
            parallel_links_per_group=schedule["group_size"],
            parallel_max_workers=schedule["workers"],
        )

    output, stats = benchmark(
        run_parallel,
        warmups=SHIFT_WARMUPS,
        repeats=SHIFT_REPEATS,
    )
    comparison = compare_link_maps(output, baseline_links, m_start=1)
    schedule_rows.append(
        {
            **schedule,
            **stats,
            "max_abs": comparison["max_abs"],
            "rel_l2": comparison["rel_l2"],
        }
    )

best_schedule = min(schedule_rows, key=lambda row: row["median_s"])

if pd is not None:
    schedule_table = pd.DataFrame(schedule_rows)
    display(schedule_table)
else:
    schedule_table = schedule_rows
    for row in schedule_rows:
        print(row)

print("Selected desktop schedule:", best_schedule["name"])


,name,group_size,workers,median_s,q25_s,q75_s,iqr_s,minimum_s,maximum_s,all_s,max_abs,rel_l2
0,materialized,6,1,1.053715,1.046995,1.063077,0.016082,1.038943,1.086062,"[1.038943258987274, 1.0469077509478666, 1.0519...",0.0,0.0
1,groups_3_workers_2,3,2,1.007596,1.000823,1.011045,0.010222,0.992560,1.015317,"[0.9925602680305019, 0.9975596879958175, 1.015...",0.0,0.0
2,groups_2_workers_3,2,3,1.011263,1.004847,1.015618,0.010770,1.000478,1.039188,"[1.0168331199674867, 1.0129236489883624, 1.026...",0.0,0.0
3,groups_1_workers_6,1,6,1.033978,1.026443,1.044980,0.018538,1.013006,1.054432,"[1.023436333052814, 1.038133212015964, 1.03370...",0.0,0.0


Selected desktop schedule: groups_3_workers_2


## 4. Complete full-year A1


In [7]:
def baseline_a1():
    context = sequential_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        execution_mode="materialized",
    )


if best_schedule["name"] == "materialized":
    def selected_a1():
        context = parallel_dwt()
        return one_arm_plan.apply_device(
            context["w_t_plus"],
            context["w_t_cross"],
            execution_mode="materialized",
        )
else:
    def selected_a1():
        context = parallel_dwt()
        return one_arm_plan.apply_device(
            context["w_t_plus"],
            context["w_t_cross"],
            execution_mode="materialized_parallel_links",
            parallel_links_per_group=int(best_schedule["group_size"]),
            parallel_max_workers=int(best_schedule["workers"]),
        )


baseline_a1_output, baseline_a1_stats = benchmark(
    baseline_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)
selected_a1_output, selected_a1_stats = benchmark(
    selected_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)
a1_comparison = compare_link_maps(
    selected_a1_output,
    baseline_a1_output,
    m_start=1,
)

print("Baseline A1:", baseline_a1_stats)
print("Selected A1:", selected_a1_stats)
print("Comparison:", a1_comparison)
print("RSS [GiB]:", f"{rss_gib():.2f}")


Baseline A1: {'median_s': 1.2684570759884082, 'q25_s': 1.2479225840070285, 'q75_s': 1.2741680939798243, 'iqr_s': 0.026245509972795844, 'minimum_s': 1.2157158059999347, 'maximum_s': 1.2823544070124626, 'all_s': [1.2741680939798243, 1.2684570759884082, 1.275971846014727, 1.2689773649908602, 1.2823544070124626, 1.2157158059999347, 1.2479225840070285, 1.2463153220014647, 1.2672019340097904]}
Selected A1: {'median_s': 1.09423332096776, 'q25_s': 1.0924853049800731, 'q75_s': 1.1096805359702557, 'iqr_s': 0.01719523099018261, 'minimum_s': 1.0795783970388584, 'maximum_s': 1.1297661809949204, 'all_s': [1.1200885429861955, 1.1297661809949204, 1.0924853049800731, 1.0975976000190713, 1.0839212519931607, 1.0795783970388584, 1.1096805359702557, 1.09423332096776, 1.0931569709791802]}
Comparison: {'per_link': {'(2, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 2)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(1, 3)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(2, 3)': {'max_abs': 0.

In [8]:
results = {
    "source": source_state,
    "machine": {
        "node": platform.node(),
        "processor": platform.processor(),
        "physical_cores": (
            psutil.cpu_count(logical=False)
            if psutil is not None
            else None
        ),
    },
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "L_trunc": L_TRUNC,
        "calc_m0": CALC_M0,
        "shift_kwargs": SHIFT_KWARGS,
    },
    "cache_files": {
        "waveform": str(waveform_cache),
        "wdm_coefficients": str(wdm_cache),
        "metadata": str(cache_metadata),
    },
    "setup_s": {
        "waveform_generation": waveform_generation_s,
        "grid": grid_s,
        "one_arm_plan": plan_s,
    },
    "dwt": {
        "sequential": sequential_dwt_stats,
        "parallel": parallel_dwt_stats,
        "plus_comparison": dwt_plus,
        "cross_comparison": dwt_cross,
    },
    "shift_schedules": schedule_rows,
    "selected_schedule": best_schedule,
    "complete_a1": {
        "baseline": baseline_a1_stats,
        "selected": selected_a1_stats,
        "comparison": a1_comparison,
    },
    "memory": {
        "plan_bytes": one_arm_plan.shift_plan.device_plan_memory_bytes,
        "rss_gib": rss_gib(),
    },
}

print(json.dumps(json_ready(results), indent=2))


if SAVE_RESULTS:
    save_json("_local_wdm_03_desktop_profile.json", results)
    if pd is not None:
        schedule_path = (
            NOTEBOOK_DIR / "_local_wdm_03_desktop_shift_schedule.csv"
        )
        pd.DataFrame(schedule_rows).to_csv(
            schedule_path,
            index=False,
        )
        print("Saved:", schedule_path)


{
  "source": {
    "root": "/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse",
    "commit": "6a5135635e74ece1beaf03beb140735534343640",
    "dirty": true
  },
  "machine": {
    "node": "calx038.ast.cam.ac.uk",
    "processor": "x86_64",
    "physical_cores": 20
  },
  "configuration": {
    "N": 3200000,
    "dt": 10.0,
    "Nf": 100,
    "L_trunc": 25,
    "calc_m0": false,
    "shift_kwargs": {
      "assembly_backend": "production",
      "assembly_precision": "complex64",
      "row_chunk_size": 32640,
      "lag_block_size": 17,
      "batch_chunk": 6,
      "tl_tp_mode": "interp",
      "tl_tp_interp_points": 32,
      "tl_tp_interp_kind": "linear"
    }
  },
  "cache_files": {
    "waveform": "/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_waveform_cache.npz",
    "wdm_coefficients": "/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_wdm_coefficients.npz",
    "metadata": "/data/tk593/LISA_Response/LISArespon